<a href="https://colab.research.google.com/github/MPashaAlv/mpashaalv/blob/main/transaksi_kotor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETL DATA TRANSAKSI KENDARAAN

In [32]:
import pandas as pd
import numpy as np
import re


## EXTRACT

In [33]:
df = pd.read_csv('transaksi_kotor.csv')

print("Jumlah data awal:", len(df))
display(df.head())

Jumlah data awal: 50


,Tanggal Terjual,Tanggal Pembelian,Kode Transaksi,Merek,Model,Transmisi,Kapasitas Mesin (CC),Tahun Produksi,Harga (Rp),Unit,Kota Transaksi
0,05-01-23,28-12-2022,NaN,Honda,BeAT,Matic,110,2023,18000000,1,Jakarta
1,12-01-2023,01/03/2023,NaN,Honda,Scoopy,matic,110,2023,21500 rb,1,Bandung
2,-,15-01-23,PB01-15,Yamaha,Fazzio,NaN,125,2023,Rp 22000000,5,Surabaya
3,"January 22, 2023",01/10/2023,TJ01_22,Honda,Vario 125,matic,125,2023,24000000,1,Medan
4,-,26-01-2023,???,Honda,Vario 160,NaN,160,2023,NaN,4,Bali


## TRANSFORM

### 1. MEMBERSIHKAN TANGGAL

In [34]:
for col in ['Tanggal Terjual', 'Tanggal Pembelian']:

    # ganti tanda "-" menjadi kosong
    df[col] = df[col].replace('-', np.nan)

    # konversi ke format tanggal
    df[col] = pd.to_datetime(
        df[col],
        errors='coerce',
        dayfirst=True
    )
    # isi tanggal kosong dengan metode forward fill
df['Tanggal Terjual'] = df['Tanggal Terjual'].fillna(method='ffill')
df['Tanggal Pembelian'] = df['Tanggal Pembelian'].fillna(method='ffill')

/tmp/ipykernel_743/443012128.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
/tmp/ipykernel_743/443012128.py:13: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Tanggal Terjual'] = df['Tanggal Terjual'].fillna(method='ffill')
/tmp/ipykernel_743/443012128.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Tanggal Pembelian'] = df['Tanggal Pembelian'].fillna(method='ffill')


### 2. MEMBERSIHKAN KODE TRANSAKSI

In [35]:
# isi kode transaksi kosong
df['Kode Transaksi'] = df['Kode Transaksi'].replace('???', np.nan)
df['Kode Transaksi'] = df['Kode Transaksi'].fillna('UNKNOWN')


3. MEMBERSIHKAN TRANSMISI

In [36]:
def bersihkan_transmisi(x):

    if pd.isna(x):
        return "Matic"

    x = str(x).strip().lower()

    if x in ['matic', 'matick', 'at']:
        return 'Matic'

    if x in ['manual', 'mnl']:
        return 'Manual'

    return 'Matic'

df['Transmisi'] = df['Transmisi'].apply(bersihkan_transmisi)

4. MEMBERSIHKAN HARGA

In [37]:
def bersihkan_harga(x):

    if pd.isna(x):
        return np.nan

    x = str(x)

    # hapus Rp, rb, titik, koma, spasi
    x = x.lower()
    x = x.replace('rp', '')
    x = x.replace('rb', '000')
    x = re.sub(r'[^0-9]', '', x)

    if x == '':
        return np.nan

    return int(x)

df['Harga (Rp)'] = df['Harga (Rp)'].apply(bersihkan_harga)

# isi harga kosong dengan median
median_harga = int(df['Harga (Rp)'].median())
df['Harga (Rp)'] = df['Harga (Rp)'].fillna(median_harga)

5. MEMBERSIHKAN TEKS

In [38]:
df['Merek'] = df['Merek'].str.title()
df['Model'] = df['Model'].str.title()
df['Kota Transaksi'] = df['Kota Transaksi'].str.title()

6. HAPUS DUPLIKAT

In [39]:
df = df.drop_duplicates()

7. FORMAT TANGGAL

In [40]:
df['Tanggal Terjual'] = df['Tanggal Terjual'].dt.strftime('%Y-%m-%d')
df['Tanggal Pembelian'] = df['Tanggal Pembelian'].dt.strftime('%Y-%m-%d')

## LOAD

In [41]:
# Simpan ke Excel
output_file = "transaksi_bersih.xlsx"

df.to_excel(output_file, index=False)

print("Data berhasil dibersihkan")
print("Jumlah data akhir:", len(df))
print("File tersimpan:", output_file)

# Download otomatis ke laptop
from google.colab import files
files.download(output_file)
display(df.head())

Data berhasil dibersihkan
Jumlah data akhir: 50
File tersimpan: transaksi_bersih.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Tanggal Terjual,Tanggal Pembelian,Kode Transaksi,Merek,Model,Transmisi,Kapasitas Mesin (CC),Tahun Produksi,Harga (Rp),Unit,Kota Transaksi
0,2023-01-05,2022-12-28,UNKNOWN,Honda,Beat,Matic,110,2023,18000000.0,1,Jakarta
1,2023-01-12,2022-12-28,UNKNOWN,Honda,Scoopy,Matic,110,2023,21500000.0,1,Bandung
2,2023-01-12,2022-12-28,PB01-15,Yamaha,Fazzio,Matic,125,2023,22000000.0,5,Surabaya
3,2023-01-22,2022-12-28,TJ01_22,Honda,Vario 125,Matic,125,2023,24000000.0,1,Medan
4,2023-01-22,2023-01-26,UNKNOWN,Honda,Vario 160,Matic,160,2023,31500000.0,4,Bali
